In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Dataset is not included in the repo
# Download link is provided in README

DATASET_PATH = "/content/data"

In [ ]:
# dataset loading
import os

DATASET_PATH = "/content/drive/MyDrive/dfd_project/clips"
REAL_DIR = os.path.join(BASE_CLIPS, "real")
FAKE_DIR = os.path.join(BASE_CLIPS, "fake")

real_clips = [f for f in os.listdir(REAL_DIR) if f.endswith(".npy")]
fake_clips = [f for f in os.listdir(FAKE_DIR) if f.endswith(".npy")]

print("CLIP COUNT SUMMARY")
print(f" REAL clips : {len(real_clips)}")
print(f" FAKE clips : {len(fake_clips)}")
print(f" TOTAL clips: {len(real_clips) + len(fake_clips)}")


CLIP COUNT SUMMARY
 REAL clips : 607
 FAKE clips : 587
 TOTAL clips: 1194


In [ ]:
# Dataset Preprocessing

import os
import numpy as np
import torch
from torch.utils.data import Dataset

class DeepfakeClipDataset(Dataset):
    def __init__(self, clips_root, transform=None):
        self.samples = []
        self.transform = transform

        real_dir = os.path.join(clips_root, "real")
        fake_dir = os.path.join(clips_root, "fake")

        for f in os.listdir(real_dir):
            if f.endswith(".npy"):
                self.samples.append((os.path.join(real_dir, f), 0))  # REAL = 0

        for f in os.listdir(fake_dir):
            if f.endswith(".npy"):
                self.samples.append((os.path.join(fake_dir, f), 1))  # FAKE = 1

        print(f"Loaded {len(self.samples)} total clips")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        clip = np.load(path)  # (T, H, W, C)
        clip = torch.tensor(clip, dtype=torch.float32)
        clip = clip.permute(3, 0, 1, 2)  # → (C, T, H, W)
        clip = clip / 255.0

        if self.transform:
            clip = self.transform(clip)

        return clip, label



In [ ]:
#training & validation
from torch.utils.data import DataLoader, random_split

CLIPS_ROOT = "/content/drive/MyDrive/dfd_project/clips"

dataset = DeepfakeClipDataset(CLIPS_ROOT)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

print(f"Train clips: {len(train_ds)} | Val clips: {len(val_ds)}")


In [ ]:
# Model defining (Used 3D CNN)
import torch.nn as nn

class SimpleTemporal3D(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),

            nn.Conv3d(64, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [ ]:
# Parameter tuning & Model Training
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleTemporal3D().to(device)
weights = torch.tensor([1.0, 1.3]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.5
)

EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for clips, labels in train_loader:
        clips = clips.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * clips.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = 100.0 * correct / total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Train Acc: {acc:.2f}%"
    )


    scheduler.step()


In [ ]:
## saving model to onnx format
import torch
import torch.onnx
import os

dummy_input = torch.randn(
    1,
    3,
    16,
    112,
    112
).to(next(model.parameters()).device)


ONNX_PATH = "/content/drive/MyDrive/dfd_project/models/deepfake_3dcnn.onnx"
os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch"},
        "logits": {0: "batch"}
    }
)

print(" ONNX model exported to:")
print(ONNX_PATH)

In [ ]:
# Model evaluation(validation accuracy)
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
import numpy as np

model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for clips, labels in val_loader:
        clips = clips.to(device)
        labels = labels.to(device)

        outputs = model(clips)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc*100:.2f}%")

In [ ]:
# model evaluation Reports
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for clips, labels in val_loader:
        clips = clips.to(device)
        outputs = model(clips)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification Report
print("\n--- Detailed Classification Report ---")
print(classification_report(all_labels, all_preds, target_names=["REAL", "FAKE"], zero_division=0))


fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Plot A: Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["REAL", "FAKE"])
disp.plot(cmap=plt.cm.Blues, ax=ax[0], colorbar=False)
ax[0].set_title("Confusion Matrix")

# Plot B: ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc_score:.3f}')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_xlabel('False Positive Rate (FPR)')
ax[1].set_ylabel('True Positive Rate (TPR)')
ax[1].set_title('ROC Curve')
ax[1].legend(loc="lower right")
ax[1].grid(alpha=0.3)

# Plot C: Precision-Recall Curve
precision, recall, _ = precision_recall_curve(all_labels, all_probs)
avg_precision = average_precision_score(all_labels, all_probs)
ax[2].plot(recall, precision, color='green', lw=2, label=f'AP = {avg_precision:.3f}')
ax[2].set_xlabel('Recall')
ax[2].set_ylabel('Precision')
ax[2].set_title('Precision-Recall Curve')
ax[2].legend(loc="lower left")
ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# !pip install -q onnx onnxscript onnxruntime (important before importing model in onnx format)

In [ ]:
## model testing code
import cv2
import numpy as np
import onnxruntime as ort
from google.colab import files

ONNX_PATH = "/content/drive/MyDrive/dfd_project/models/deepfake_3dcnn.onnx"
session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])

def preprocess_video(video_path, num_frames=16, size=(112, 112)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, size)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()


    video_data = np.array(frames).astype(np.float32)
    video_data = video_data / 255.0
    video_data = np.transpose(video_data, (3, 0, 1, 2))
    video_data = np.expand_dims(video_data, axis=0)

    return video_data


print("Upload a video for testing:")
uploaded = files.upload()

for filename in uploaded.keys():

    input_tensor = preprocess_video(filename)


    inputs = {session.get_inputs()[0].name: input_tensor}
    outputs = session.run(None, inputs)


    logits = outputs[0]
    exp_logits = np.exp(logits - np.max(logits))
    probs = exp_logits / exp_logits.sum()

    prediction = np.argmax(probs)
    label = "FAKE" if prediction == 1 else "REAL"
    confidence = probs[0][prediction] * 100

    print(f"\n--- Result for {filename} ---")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")